In [ ]:
library(vegan)      # For RDA and ordination
library(ggplot2)    # For plotting
library(ggrepel)    # For non-overlapping labels in plots
library(tidyverse)  # For data manipulation (dplyr, etc.)
library(e1071)      # Skewness calc

In [ ]:
df <- read.csv("C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_uscore_reduced.csv") #nolint 
# Remove Upland class
df <- df[df$Class != "UL", ]

# remove Hornpeak sites because we are using WTP data now
hornpeak <- c("HPA", "HPB", "HPC", "HPD", "HPE")
df <- df[!df$SiteID %in% hornpeak, ]

names(df)

In [ ]:
# ============================================================================
# STEP 2: Define variable groups (column ranges)
# ============================================================================

# Vegetation UAV ALL (predictor RDA1)
#veg_start <- which(names(df) == "REDEDGE_min")  
#veg_end   <- which(names(df) == "stem_density")      
#veg_cols <- names(df)[veg_start:veg_end]

# Veg - UAV structure only
struct_start <- which(names(df)=="average_veg_height")
struct_end   <- which(names(df) == "stem_density")
struct_cols  <- names(df)[struct_start:struct_end]

# Veg - UAV MS only
ms_start <- which(names(df)=="NDVI_MAX")
ms_end   <- which(names(df) == "NIR_median")
ms_cols <- names(df)[ms_start:ms_end]
# Remove SR_MAX
ms_cols <- ms_cols[ms_cols != "SR_MAX"]

# Field veg 
field_start <- which(names(df)=="small_tree")
field_end   <- which(names(df)=="equisetum_dom")
field_cols <- names(df)[field_start:field_end]

# Hydrology (predictors for RDA1, response for RDA2)
hydro_start <- which(names(df) == "WTP_max")      
hydro_end   <- which(names(df) == "VHG_range")  
hydro_cols <- names(df)[hydro_start:hydro_end]
#hydro_cols <- hydro_cols[!hydro_cols %in% c("WTP_range", "WTP_stdev")]
hydro_cols <- hydro_cols[!hydro_cols %in% c("unknown_hydro")]

# Chemistry (predictors for RDA1, response for RDA2)
chem_start <- which(names(df) == "Al")
chem_end   <- which(names(df) == "ORPmV")
chem_cols <- names(df)[chem_start:chem_end]
chem_cols <- chem_cols[chem_cols != "TDS_mg.L"]

# UPDATE: Use only non-censored chemistry variables
#noncensored_chem <- c("Ba", "Ca", "Fe", "Mg", "Mn", "Si", "Na", "Sr","NH3_total", "P_total", "Hardness", "Dissolved_O_percent", "SpC", "pH", "ORPmV")
#chem_cols <- chem_cols[chem_cols %in% noncensored_chem]


# Topography (predictors for RDA2)
#topo_start <- which(names(df) == "SWI_median")
#topo_end   <- which(names(df) == "Geom_mode_10")
#topo_cols <- names(df)[topo_start:topo_end]

# stat remotes
sat_start <- which(names(df) == "NDVI_amp_harmonic")
sat_end <- which(names(df) == "TCW_p90")

# UAV structure & indices (for reference/additional analysis if needed)
#uav_veg_cols <- c("median_veg_height", "average_veg_height", "NDVI_MEAN", "NDGVI_MEAN")  # adjust

In [ ]:
print(ms_cols)

In [ ]:
print(hydro_cols)

In [ ]:
print(chem_cols)

## NEW: Remove redunant information

In [ ]:
# ============================================================================
# STEP 2b: Remove redundant predictor information 
# (KEEP medians, DROP means + ranges for MS and structure)
# ============================================================================

# Backup original predictor lists (optional for debugging)
ms_cols_full     <- ms_cols
struct_cols_full <- struct_cols

# ---------------------------------------------------------------------------
# 1) DROP MEAN-type MS / band predictors (keep MEDIAN versions instead)
# ---------------------------------------------------------------------------
mean_like_ms <- ms_cols[grepl("_MEAN|mean", ms_cols, ignore.case = TRUE)]
ms_cols <- setdiff(ms_cols, mean_like_ms)

# ---------------------------------------------------------------------------
# 2) DROP RANGE-type MS / band predictors (e.g., NDVI_RANGE, SR_RANGE)
# ---------------------------------------------------------------------------
range_like_ms <- ms_cols[grepl("_RANGE|range", ms_cols, ignore.case = TRUE)]
ms_cols <- setdiff(ms_cols, range_like_ms)

# ---------------------------------------------------------------------------
# 3) STRUCTURE: Drop average_veg_height (redundant with median)
# ---------------------------------------------------------------------------
redundant_struct <- intersect(struct_cols, c("average_veg_height"))
struct_cols <- setdiff(struct_cols, redundant_struct)

# ---------------------------------------------------------------------------
# 4) OPTIONAL diagnostic prints
# ---------------------------------------------------------------------------
cat("\nDropped MEAN-type MS predictors:\n")
print(mean_like_ms)

cat("\nDropped RANGE-type MS predictors:\n")
print(range_like_ms)

cat("\nDropped redundant structure predictors:\n")
print(redundant_struct)

cat("\n\nFinal MS predictor set:\n")
print(ms_cols)

cat("\nFinal Structure predictor set:\n")
print(struct_cols)


In [ ]:
# ============================================================================
# STEP 4: Prepare Response Data (Chemistry + Hydrology)
# ============================================================================

# Remove VHG and WTP columns from hydro_cols
#hydro_cols <- hydro_cols[!grepl("VHG|WTP", hydro_cols)]

## NEW: Keep WTP, Drop VHG ALD is a proxy for Bog WTP vars
#hydro_cols <- hydro_cols[!grepl("VHG", hydro_cols)]
# Remove marsh wetlands since they don't have VHG data
df <- df[df$Class != "Marsh", ]
# Remove all sites that START with "DCB"
df <- df[!grepl("^DCB", df$SiteID), ]
df$Class

# Get hydrology data
hydro_data <- df[, hydro_cols]
rownames(hydro_data) <- df$SiteID

# Get chemistry data (already u-score transformed in the CSV)
chem_data <- df[, chem_cols]
rownames(chem_data) <- df$SiteID

#get field veg? 
field_data <- df[, field_cols]
rownames(field_data) <- df$SiteID

# Combine chemistry (u-score dataframe) with hydrology data as RESPONSE
response <- cbind(chem_data, hydro_data)
rownames(response) <- df$SiteID

# ============================================================================
# STEP 5: Prepare Predictor Data (Vegetation)
# ============================================================================

predictors <- df[, c(ms_cols, struct_cols)]
rownames(predictors) <- df$SiteID

# NEW: Remove highly cor or no variation vars
apply(predictors, 2, function(x) length(unique(x)))
which(abs(cor(predictors, use="pairwise.complete.obs")) > 0.99999, arr.ind = TRUE)
# 1. Find zero-variance or all-NA predictors
pred_sds <- apply(predictors, 2, sd, na.rm = TRUE)

zero_var_preds <- names(pred_sds[is.na(pred_sds) | pred_sds == 0])
print( zero_var_preds)
# Look at this list – these are variables that are constant or all NA

# 2. Drop them from predictors
predictors <- predictors[, !(names(predictors) %in% zero_var_preds), drop = FALSE]

cm <- cor(predictors, use = "pairwise.complete.obs")
diag(cm) <- NA

perfect_pairs <- which(abs(cm) == 1, arr.ind = TRUE)
perfect_pairs




In [ ]:
# ============================================================================
# STEP 6: Fit RDA1
# ============================================================================
rda_global <- rda(response ~ ., data = predictors, scale  = TRUE)
summary(rda_global)

In [ ]:
# --- VIFs (for diagnostics) ---
# vif.cca() returns VIF; you want sqrt(VIF) for your rule (sqrt(VIF) > 2)
vif_global        <- vif.cca(rda_global)
sqrt_vif_global   <- sqrt(vif_global)

cat("\nRaw VIF values (global model):\n")
print(vif_global)

cat("\nSqrt(VIF) values (global model):\n")
print(sqrt_vif_global)

# Sort by sqrt(VIF), descending
sqrt_vif_sorted <- sort(sqrt_vif_global, decreasing = TRUE)
cat("\nSqrt(VIF), sorted (descending):\n")
print(sqrt_vif_sorted)

# Identify variables with high VIF according to your rule: sqrt(VIF) > 2  (=> VIF > 4)
high_vif_global <- sqrt_vif_global[sqrt_vif_global > 2]
cat("\nVariables with sqrt(VIF) > 2 (i.e. VIF > 4):\n")
print(high_vif_global)

In [ ]:
# ============================================================================
# STEP 7: Model Statistics
# ============================================================================

# Adjusted R-squared
r2_global <- RsquareAdj(rda_global)
print(r2_global)

# Overall model significance
anova_global <- anova.cca(rda_global, permutations = 999)
print(anova_global)

# Test each axis
anova.cca(rda_global, by = "axis", permutations = 999)

# Test each predictor variable
anova.cca(rda_global, by = "term", permutations = 999)

## Plot Global RDA

In [ ]:
# ============================================================================
# STEP 8: Extract Scores for Plotting - FLEXIBLE VERSION
# ============================================================================

# SET WHICH AXES TO PLOT HERE 
axis_x <- 1  # Change 
axis_y <- 2  

# Site scores
site_scores <- as.data.frame(scores(rda_global, display = "sites", scaling = 2, choices = c(axis_x, axis_y)))
site_scores$SiteID <- rownames(site_scores)
site_scores$Class <- df$Class  # Add wetland class for coloring

# ADD HYDROLOGICAL REGIME INFORMATION
site_scores$Recharge <- df$Recharge
site_scores$Discharge <- df$Discharge
site_scores$unknown_hydro <- df$unknown_hydro

# Create a categorical variable for hydro regime
site_scores$HydroRegime <- case_when(
  site_scores$Recharge == 1 ~ "Recharge",
  site_scores$Discharge == 1 ~ "Discharge",
  site_scores$unknown_hydro == 1 ~ "Unknown",
  TRUE ~ "Unknown"  # Fallback
)

# Convert to factor with specific order
site_scores$HydroRegime <- factor(site_scores$HydroRegime, 
                                   levels = c("Recharge", "Discharge", "Unknown"))

# Predictor arrows (vegetation variables)
arrow_scores <- as.data.frame(scores(rda_global, display = "bp", scaling = 2, choices = c(axis_x, axis_y)))
arrow_scores$Variable <- rownames(arrow_scores)

# Response arrows (hydrogeochemistry variables)
response_scores <- as.data.frame(scores(rda_global, display = "species", scaling = 2, choices = c(axis_x, axis_y)))
response_scores$Variable <- rownames(response_scores)

# Get axis labels with variance explained
eig <- summary(rda_global)$concont$importance[2, ]  # Get all axes
xlab_pct <- paste0("RDA", axis_x, " (", round(eig[axis_x] * 100, 1), "%)")
ylab_pct <- paste0("RDA", axis_y, " (", round(eig[axis_y] * 100, 1), "%)")

# ============================================================================
# STEP 9: Scale arrows for visibility
# ============================================================================
# Scale vegetation predictors (red arrows)
arrow_scale_veg <- 4
arrow_scores$RDA_X_scaled <- arrow_scores[, 1] * arrow_scale_veg  # Use column index instead of name
arrow_scores$RDA_Y_scaled <- arrow_scores[, 2] * arrow_scale_veg

# Scale hydrogeochemistry response (black arrows)
arrow_scale_response <- 2
response_scores$RDA_X_scaled <- response_scores[, 1] * arrow_scale_response
response_scores$RDA_Y_scaled <- response_scores[, 2] * arrow_scale_response

# ============================================================================
# STEP 10: Triplot with Shape Mapping
# ============================================================================
# Create subtitle with adjusted R² and p-value
subtitle_text <- sprintf("Adj. R² = %.3f, p = %.3f (Axes: RDA%d vs RDA%d)", 
                        r2_global$adj.r.squared, 
                        anova_global$`Pr(>F)`[1],
                        axis_x, axis_y)

ggplot() +
  # Sites as points, colored by wetland class, shaped by hydro regime
  geom_point(data = site_scores, 
             aes(x = .data[[paste0("RDA", axis_x)]], 
                 y = .data[[paste0("RDA", axis_y)]], 
                 color = Class, 
                 shape = HydroRegime), 
             size = 3, alpha = 0.7, stroke = 1) +
  
  # Define shape mapping: triangle = recharge, square = discharge, circle = unknown
  scale_shape_manual(
    name = "Hydrological Regime",
    values = c("Recharge" = 17,      # filled triangle
               "Discharge" = 15,      # filled square
               "Unknown" = 16),       # filled circle
    labels = c("Recharge", "Discharge", "Unknown")
  ) +
  
  # Site labels
  geom_text_repel(data = site_scores, 
                  aes(x = .data[[paste0("RDA", axis_x)]], 
                      y = .data[[paste0("RDA", axis_y)]], 
                      label = SiteID),
                  color = "darkblue", size = 2, max.overlaps = 10) +
  
  # Vegetation predictor arrows (RED)
  geom_segment(data = arrow_scores,
               aes(x = 0, y = 0, xend = RDA_X_scaled, yend = RDA_Y_scaled),
               arrow = arrow(length = unit(0.25, "cm")), 
               color = "darkred", linewidth = 0.2, alpha = 0.8) +
  
  # Vegetation predictor labels
  geom_text_repel(data = arrow_scores,
                  aes(x = RDA_X_scaled, y = RDA_Y_scaled, label = Variable),
                  color = "darkred", size = 2.2,
                  max.overlaps = 200,
                  segment.color = NA,
                  box.padding = 0.2) +
  
  # Hydrogeochemistry response arrows (BLACK)
  geom_segment(data = response_scores,
               aes(x = 0, y = 0, xend = RDA_X_scaled, yend = RDA_Y_scaled),
               arrow = arrow(length = unit(0.2, "cm")), 
               color = "black", linewidth = 0.2, alpha = 0.8) +
  
  # Hydrogeochemistry response labels
  geom_text_repel(data = response_scores,
                  aes(x = RDA_X_scaled, y = RDA_Y_scaled, label = Variable),
                  color = "black", size = 2,
                  max.overlaps = 500,
                  segment.color = NA) +
  
  # Reference lines
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  
  labs(
    title = "RDA1: Hydro + chem ~ UAV MS and structure (VHG inlcuded)",
    subtitle = subtitle_text,
    x = xlab_pct,
    y = ylab_pct,
    color = "Wetland Class"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
    panel.grid.minor = element_blank(),
    axis.line = element_line(color = "grey40", linewidth = 0.3),
    legend.position = "right",
    legend.box = "vertical"  # Stack the two legends vertically
  )

# NEW: Forward selection

In [ ]:
set.seed(123)

mod0 <- rda(response ~ 1, predictors, scale = TRUE) # model with intercept only 
mod1 <- rda(response ~ ., predictors, scale = TRUE) # model with all explanatory variables

perm_ctrl <- how(nperm = 999) # permutation settings

# 4. Forward selection using adjusted R2 (Blanchet method)
mod_forward <- ordiR2step(
  object         = mod0,        # start with null
  scope          = mod1,        # full set of predictors
  Pin            = 0.01,        # mod strict alpha
  R2scope        = TRUE,        # do not exceed global adjR2
  permutations   = perm_ctrl,
  R2permutations = 999,
  trace          = TRUE
)

In [ ]:
RsquareAdj(mod_forward)      # final adjusted R²
anova(mod_forward)           # overall significance
anova(mod_forward, by = "axis")  # axis-wise tests
anova(mod_forward, by = "term")  # each selected predictor


# Re-Run RDA with forward selected vars

In [ ]:
# ==============================
# STEP 2: Final RDA model
# ==============================

# 1) Extract selected predictors from the full predictor matrix
selected_vars <- attr(terms(mod_forward), "term.labels")
selected_vars

predictors_sel <- predictors[, selected_vars, drop = FALSE]

# 2) Refit RDA using only selected predictors (optional but clear)
#rda_final <- rda(response ~ ., data = predictors_sel, scale = TRUE)

# If you prefer, you can just do:
rda_final <- mod_forward
summary(rda_final)

# ==============================
# STEP 3: Model statistics
# ==============================

# Adjusted R²
r2_final <- RsquareAdj(rda_final)
r2_final  # $r.squared and $adj.r.squared

# Overall model significance
anova_final <- anova.cca(rda_final, permutations = 999)
anova_final

# Test each axis
anova_axes <- anova.cca(rda_final, by = "axis", permutations = 999)
anova_axes

# Test each predictor variable
anova_terms <- anova.cca(rda_final, by = "term", permutations = 999)
anova_terms


## Plot reduced var RDA

In [ ]:
# ==============================
# STEP 4: Extract scores
# ==============================

# Site scores
site_scores <- as.data.frame(scores(rda_final, display = "sites", scaling = 2))
site_scores$SiteID <- rownames(site_scores)
site_scores$Class  <- df$Class  # wetland class for coloring

# ADD HYDROLOGICAL REGIME INFORMATION
site_scores$Recharge <- df$Recharge
site_scores$Discharge <- df$Discharge
site_scores$unknown_hydro <- df$unknown_hydro

# Create a categorical variable for hydro regime
site_scores$HydroRegime <- case_when(
  site_scores$Recharge == 1 ~ "Recharge",
  site_scores$Discharge == 1 ~ "Discharge",
  site_scores$unknown_hydro == 1 ~ "Unknown",
  TRUE ~ "Unknown"  # Fallback
)

# Convert to factor with specific order
site_scores$HydroRegime <- factor(site_scores$HydroRegime, 
                                   levels = c("Recharge", "Discharge", "Unknown"))

# Predictor arrows (vegetation variables = constrained predictors)
arrow_scores <- as.data.frame(scores(rda_final, display = "bp", scaling = 2))
arrow_scores$Variable <- rownames(arrow_scores)

# Response arrows (hydro + chem = 'species' in vegan's language)
response_scores <- as.data.frame(scores(rda_final, display = "species", scaling = 2))
response_scores$Variable <- rownames(response_scores)

# ==============================
# Axis labels with variance explained
# ==============================

eig <- summary(rda_final)$concont$importance[2, 1:2]  # proportion of constrained variance for RDA1 & RDA2
xlab_pct <- paste0("RDA1 (", round(eig[1] * 100, 1), "%)")
ylab_pct <- paste0("RDA2 (", round(eig[2] * 100, 1), "%)")

# ==============================
# Scale arrows for visibility
# ==============================

# Vegetation predictors (red arrows)
arrow_scale_veg <- 3
arrow_scores$RDA1_scaled <- arrow_scores$RDA1 * arrow_scale_veg
arrow_scores$RDA2_scaled <- arrow_scores$RDA2 * arrow_scale_veg

# Hydro + chem responses (black arrows)
arrow_scale_response <- 2
response_scores$RDA1_scaled <- response_scores$RDA1 * arrow_scale_response
response_scores$RDA2_scaled <- response_scores$RDA2 * arrow_scale_response

# ==============================
# Subtitle: adjusted R² + model p-value
# ==============================

subtitle_text <- sprintf(
  "Forward-selected predictors (Pin = 0.01). Adj. R² = %.3f, p = %.3f",
  r2_final$adj.r.squared,
  anova_final$`Pr(>F)`[1]
)

# ==============================
# Triplot with Shape Mapping
# ==============================

ggplot() +
  # Sites as points, colored by wetland class, shaped by hydro regime
  geom_point(
    data = site_scores,
    aes(x = RDA1, y = RDA2, color = Class, shape = HydroRegime),
    size = 3, alpha = 0.7, stroke = 1
  ) +
  
  # Define shape mapping: triangle = recharge, square = discharge, circle = unknown
  scale_shape_manual(
    name = "Hydrological Regime",
    values = c("Recharge" = 17,      # filled triangle
               "Discharge" = 15,      # filled square
               "Unknown" = 16),       # filled circle
    labels = c("Recharge", "Discharge", "Unknown")
  ) +
  
  # Site labels
  geom_text_repel(
    data = site_scores,
    aes(x = RDA1, y = RDA2, label = SiteID),
    color = "darkblue",
    size = 2,
    max.overlaps = 10
  ) +
  
  # Vegetation predictor arrows (RED)
  geom_segment(
    data = arrow_scores,
    aes(x = 0, y = 0, xend = RDA1_scaled, yend = RDA2_scaled),
    arrow = arrow(length = unit(0.25, "cm")),
    color = "darkred",
    linewidth = 0.2,
    alpha = 0.8
  ) +
  
  # Vegetation predictor labels
  geom_text_repel(
    data = arrow_scores,
    aes(x = RDA1_scaled, y = RDA2_scaled, label = Variable),
    color = "darkred",
    size = 2.2,
    max.overlaps = 50,
    segment.color = NA,
    box.padding = 0.2
  ) +
  
  # Hydrogeochemistry response arrows (BLACK)
  geom_segment(
    data = response_scores,
    aes(x = 0, y = 0, xend = RDA1_scaled, yend = RDA2_scaled),
    arrow = arrow(length = unit(0.2, "cm")),
    color = "black",
    linewidth = 0.2,
    alpha = 0.8
  ) +
  
  # Hydrogeochemistry response labels
  geom_text_repel(
    data = response_scores,
    aes(x = RDA1_scaled, y = RDA2_scaled, label = Variable),
    color = "black",
    size = 2,
    max.overlaps = 200,
    segment.color = NA
  ) +
  
  # Reference lines
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  
  labs(
    title    = "RDA1: Hydro + chemistry ~ forward-selected UAV MS + structure (using VHG)",
    subtitle = subtitle_text,
    x = xlab_pct,
    y = ylab_pct,
    color = "Wetland Class"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
    panel.grid.minor = element_blank(),
    axis.line        = element_line(color = "grey40", linewidth = 0.3),
    legend.position  = "right",
    legend.box = "vertical"  # Stack the two legends vertically
  )

In [ ]:
# Predictor (constraining) scores
pred_scores <- as.data.frame(scores(rda_final, display = "bp", scaling = 2))
pred_scores$Variable <- rownames(pred_scores)

# Top N to show
N <- 10

# Top predictors for RDA1 (by absolute loading)
top_pred_RDA1 <- pred_scores[order(-abs(pred_scores$RDA1)), c("Variable", "RDA1")][1:N, ]
top_pred_RDA1

# Top predictors for RDA2
top_pred_RDA2 <- pred_scores[order(-abs(pred_scores$RDA2)), c("Variable", "RDA2")][1:N, ]
top_pred_RDA2


In [ ]:
# Response scores (hydro + chemistry)
resp_scores <- as.data.frame(scores(rda_final, display = "species", scaling = 2))
resp_scores$Variable <- rownames(resp_scores)

N <- 8  # top 8

# Top responses for RDA1
top_resp_RDA1 <- resp_scores[order(-abs(resp_scores$RDA1)), c("Variable", "RDA1")][1:N, ]
top_resp_RDA1

# Top responses for RDA2
top_resp_RDA2 <- resp_scores[order(-abs(resp_scores$RDA2)), c("Variable", "RDA2")][1:N, ]
top_resp_RDA2


## Kaiser–Guttman criterion 

In [ ]:
# Show me all the residual axes that have eigenvalues bigger than the average residual eigenvalue
# leftover gradients in the data that are not explained by the predictors
rda_final$CA$eig[rda_final$CA$eig > mean(rda_final$CA$eig)]
rda_global$CA$eig[rda_global$CA$eig > mean(rda_global$CA$eig)]


# PCA for Field Veg and Hydrochem

In [ ]:
# ============================================================================
# PCA: Field Vegetation + Hydrology + Chemistry
# ============================================================================

# ============================================================================
# CONFIGURATION: Set this to include/exclude VHG data
# ============================================================================
INCLUDE_VHG <- TRUE  # Change to FALSE to exclude VHG data

# ============================================================================
# STEP 1: Load and initial filtering
# ============================================================================
df <- read.csv("C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_uscore_reduced.csv")

# Remove Upland class
df <- df[df$Class != "UL", ]

# Remove Hornpeak sites
hornpeak <- c("HPA", "HPB", "HPC", "HPD", "HPE")
df <- df[!df$SiteID %in% hornpeak, ]

# ============================================================================
# STEP 2: VHG-dependent filtering
# ============================================================================
if (INCLUDE_VHG) {
  cat("\n=== VHG DATA INCLUDED ===\n")
  cat("Filtering out Marsh class and DCB sites...\n")
  
  # Remove Marsh class
  df <- df[df$Class != "M", ]
  
  # Remove DCB sites
  df <- df[!grepl("^DCB", df$SiteID), ]
  
  cat("Sites remaining:", nrow(df), "\n")
  cat("Classes:", paste(unique(df$Class), collapse = ", "), "\n")
} else {
  cat("\n=== VHG DATA EXCLUDED ===\n")
  cat("All wetland classes included\n")
}

# ============================================================================
# STEP 3: Save hydro regime columns for symbology (NOT for PCA)
# ============================================================================
df$Recharge_regime <- df$Recharge
df$Discharge_regime <- df$Discharge
df$unknown_hydro_regime <- df$unknown_hydro

# ============================================================================
# STEP 4: Define variable groups
# ============================================================================

# Field vegetation
field_start <- which(names(df) == "small_tree")
field_end   <- which(names(df) == "lichen")
field_cols <- names(df)[field_start:field_end]

# Hydrology
hydro_start <- which(names(df) == "WTP_max")
hydro_end   <- which(names(df) == "VHG_range")
hydro_cols <- names(df)[hydro_start:hydro_end]

# Remove ordinal hydro regime indicators from analysis
# (keep them in df for symbology, just don't include in PCA)
hydro_cols <- hydro_cols[!hydro_cols %in% c("Recharge", "Discharge", "unknown_hydro")]

# Conditionally include/exclude VHG
if (!INCLUDE_VHG) {
  hydro_cols <- hydro_cols[!grepl("^VHG", hydro_cols)]
  cat("VHG variables excluded from analysis\n")
} else {
  cat("VHG variables included in analysis\n")
}

# Optional: Remove WTP_range and WTP_stdev if desired
hydro_cols <- hydro_cols[!hydro_cols %in% c("WTP_range", "WTP_stdev")]

# Chemistry
chem_start <- which(names(df) == "Al")
chem_end   <- which(names(df) == "ORPmV")
chem_cols <- names(df)[chem_start:chem_end]
chem_cols <- chem_cols[chem_cols != "TDS_mg.L"]

cat("\nVariables in PCA:\n")
cat("Field veg:", length(field_cols), "\n")
cat("Hydrology:", length(hydro_cols), "\n")
cat("Chemistry:", length(chem_cols), "\n")

# ============================================================================
# STEP 5: Prepare PCA matrix and handle missing data
# ============================================================================

# Combine all variables for PCA
pca_vars <- df[, c(field_cols, hydro_cols, chem_cols)]

# Remove rows with any NA values
complete_cases <- complete.cases(pca_vars)
if (sum(!complete_cases) > 0) {
  cat("\nRemoving", sum(!complete_cases), "sites with missing data:\n")
  cat("Sites removed:", df$SiteID[!complete_cases], "\n")
  
  pca_vars <- pca_vars[complete_cases, ]
  df <- df[complete_cases, ]
}

cat("\nFinal dataset:", nrow(pca_vars), "sites\n")

# Scale everything before PCA
pca_scaled <- scale(pca_vars)

# Run PCA
pca_all <- rda(pca_scaled)

# ============================================================================
# STEP 6: Check variance explained and scree plot
# ============================================================================
screeplot(pca_all, bstick = TRUE)

# Get variance explained
pca_summary <- summary(pca_all)
eigenvalues <- pca_summary$cont$importance

cat("\nPC1 explains:", round(eigenvalues[2,1] * 100, 2), "% of variance\n")
cat("PC2 explains:", round(eigenvalues[2,2] * 100, 2), "% of variance\n")
cat("PC1+PC2 together explain:", round(eigenvalues[3,2] * 100, 2), "% of variance\n")

# ============================================================================
# STEP 7: Extract scores for plotting
# ============================================================================
site_scores <- as.data.frame(scores(pca_all, display = "sites", choices = 1:2))
site_scores$SiteID <- df$SiteID
site_scores$Class <- df$Class

# ADD HYDROLOGICAL REGIME INFORMATION (for symbology only)
site_scores$Recharge <- df$Recharge_regime
site_scores$Discharge <- df$Discharge_regime
site_scores$unknown_hydro <- df$unknown_hydro_regime

# Create categorical variable for hydro regime
site_scores$HydroRegime <- case_when(
  site_scores$Recharge == 1 ~ "Recharge",
  site_scores$Discharge == 1 ~ "Discharge",
  site_scores$unknown_hydro == 1 ~ "Unknown",
  TRUE ~ "Unknown"
)

site_scores$HydroRegime <- factor(site_scores$HydroRegime, 
                                   levels = c("Recharge", "Discharge", "Unknown"))

# Variable loadings
var_scores <- as.data.frame(scores(pca_all, display = "species", choices = 1:2))
var_scores$Variable <- rownames(var_scores)

# Get axis labels
xlab_pct <- paste0("PC1 (", round(eigenvalues[2,1] * 100, 1), "%)")
ylab_pct <- paste0("PC2 (", round(eigenvalues[2,2] * 100, 1), "%)")

# ============================================================================
# STEP 8: Identify top contributing variables
# ============================================================================
var_scores$contrib_PC1 <- abs(var_scores$PC1)
var_scores$contrib_PC2 <- abs(var_scores$PC2)
var_scores$total_contrib <- sqrt(var_scores$PC1^2 + var_scores$PC2^2)

top_PC1 <- var_scores %>%
  arrange(desc(contrib_PC1)) %>%
  slice_head(n = 10) %>%
  select(Variable, PC1, contrib_PC1)

top_PC2 <- var_scores %>%
  arrange(desc(contrib_PC2)) %>%
  slice_head(n = 10) %>%
  select(Variable, PC2, contrib_PC2)

cat("\nTop 10 variables on PC1:\n")
print(top_PC1)

cat("\nTop 10 variables on PC2:\n")
print(top_PC2)

# Select top 20 for plotting
top_vars <- var_scores %>%
  arrange(desc(total_contrib)) %>%
  slice_head(n = 20) %>%
  pull(Variable)

var_scores_top <- var_scores %>%
  filter(Variable %in% top_vars)

# ============================================================================
# STEP 9: Scale arrows for visibility
# ============================================================================
arrow_scale <- 3
var_scores_top$PC1_scaled <- var_scores_top$PC1 * arrow_scale
var_scores_top$PC2_scaled <- var_scores_top$PC2 * arrow_scale

# ============================================================================
# STEP 10: Create PCA biplot (Top 20 variables)
# ============================================================================
plot_title <- ifelse(INCLUDE_VHG,
                     "PCA: Field Veg + Hydrology (with VHG) + Chemistry",
                     "PCA: Field Veg + Hydrology (no VHG) + Chemistry")

ggplot() +
  geom_point(data = site_scores, 
             aes(x = PC1, y = PC2, color = Class, shape = HydroRegime), 
             size = 3, alpha = 0.7, stroke = 1) +
  
  scale_shape_manual(
    name = "Hydrological Regime",
    values = c("Recharge" = 17, "Discharge" = 15, "Unknown" = 16),
    labels = c("Recharge", "Discharge", "Unknown")
  ) +
  
  geom_text_repel(data = site_scores, 
                  aes(x = PC1, y = PC2, label = SiteID),
                  color = "darkblue", size = 2, max.overlaps = 10) +
  
  geom_segment(data = var_scores_top,
               aes(x = 0, y = 0, xend = PC1_scaled, yend = PC2_scaled),
               arrow = arrow(length = unit(0.25, "cm")), 
               color = "grey30", linewidth = 0.2, alpha = 0.7) +
  
  geom_text_repel(data = var_scores_top,
                  aes(x = PC1_scaled, y = PC2_scaled, label = Variable),
                  color = "grey20", size = 2.2,
                  max.overlaps = 50,
                  segment.color = NA,
                  box.padding = 0.2) +
  
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  
  labs(
    title = paste0(plot_title, " (Top 20 Variables)"),
    x = xlab_pct,
    y = ylab_pct,
    color = "Wetland Class"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
    panel.grid.minor = element_blank(),
    axis.line = element_line(color = "grey40", linewidth = 0.3),
    legend.position = "right",
    legend.box = "vertical"
  )

# ============================================================================
# STEP 11: Biplot with ALL variables
# ============================================================================
var_scores_all <- var_scores
var_scores_all$PC1_scaled <- var_scores_all$PC1 * arrow_scale
var_scores_all$PC2_scaled <- var_scores_all$PC2 * arrow_scale

ggplot() +
  geom_point(data = site_scores, 
             aes(x = PC1, y = PC2, color = Class, shape = HydroRegime), 
             size = 3, alpha = 0.7, stroke = 1) +
  
  scale_shape_manual(
    name = "Hydrological Regime",
    values = c("Recharge" = 17, "Discharge" = 15, "Unknown" = 16),
    labels = c("Recharge", "Discharge", "Unknown")
  ) +
  
  geom_text_repel(data = site_scores, 
                  aes(x = PC1, y = PC2, label = SiteID),
                  color = "darkblue", size = 2, max.overlaps = 10) +
  
  geom_segment(data = var_scores_all,
               aes(x = 0, y = 0, xend = PC1_scaled, yend = PC2_scaled),
               arrow = arrow(length = unit(0.2, "cm")), 
               color = "grey40", linewidth = 0.15, alpha = 0.4) +
  
  geom_text_repel(data = var_scores_all,
                  aes(x = PC1_scaled, y = PC2_scaled, label = Variable),
                  color = "grey20", size = 1.8,
                  max.overlaps = 200,
                  segment.color = NA,
                  box.padding = 0.1) +
  
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.3) +
  
  labs(
    title = paste0(plot_title, " (All Variables)"),
    x = xlab_pct,
    y = ylab_pct,
    color = "Wetland Class"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    panel.grid.major = element_line(color = "grey90", linewidth = 0.2),
    panel.grid.minor = element_blank(),
    axis.line = element_line(color = "grey40", linewidth = 0.3),
    legend.position = "right",
    legend.box = "vertical"
  )